In [2]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

ledger   = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
inv      = pd.read_csv(BASE_PATH + 'inventory_master.csv')
po_h     = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l     = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
so_h     = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l     = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')
branches = pd.read_csv(BASE_PATH + 'branches.csv')

for name, df in [('ledger', ledger), ('inv', inv), ('po_h', po_h), ('po_l', po_l),
                 ('so_h', so_h), ('so_l', so_l), ('products', products), ('branches', branches)]:
    print(name, df.shape)

Mounted at /content/drive
ledger (237230, 9)
inv (180, 8)
po_h (24000, 10)
po_l (155495, 9)
so_h (20000, 11)
so_l (130402, 9)
products (30, 23)
branches (6, 13)


In [3]:
print('inv (inventory master) rows:', len(inv))
print(inv[['opening_stock', 'reorder_level', 'safety_stock', 'max_stock', 'current_stock']].describe().round(1))

print()
print('products rows:', len(products))
print(products[['unit_cost', 'unit_price', 'reorder_level', 'safety_stock', 'max_stock_level', 'lead_time_days']].describe().round(1))

inv (inventory master) rows: 180
       opening_stock  reorder_level  safety_stock  max_stock  current_stock
count          180.0          180.0         180.0      180.0          180.0
mean           183.2           64.6          45.2      321.8       107240.4
std             63.4           24.1          16.9      116.6         5341.1
min             80.0           24.0          16.0      131.0        88853.0
25%            129.0           44.0          31.0      230.0       103487.5
50%            178.5           62.0          43.5      310.5       107790.5
75%            234.5           82.2          58.0      415.5       110779.5
max            300.0          119.0          85.0      582.0       121013.0

products rows: 30
       unit_cost  unit_price  reorder_level  safety_stock  max_stock_level  \
count       30.0        30.0           30.0          30.0             30.0   
mean     14585.0     18593.7           39.7          20.1            104.8   
std      21478.8     26542.4  

In [4]:
inv_max = inv.groupby('product_id')['max_stock'].mean().round(1)
cmp = products.set_index('product_id')[['max_stock_level']].copy()
cmp['inv_max_stock_avg'] = inv_max
cmp['ratio_inv_to_products'] = (cmp['inv_max_stock_avg'] / cmp['max_stock_level']).round(2)

print('products rows:', len(cmp), '| inv rows:', len(inv))
print(cmp.sort_values('max_stock_level').to_string())
print()
print('ratio min / median / max:', cmp['ratio_inv_to_products'].min(),
      cmp['ratio_inv_to_products'].median(), cmp['ratio_inv_to_products'].max())

products rows: 30 | inv rows: 180
            max_stock_level  inv_max_stock_avg  ratio_inv_to_products
product_id                                                           
P015                      8              305.3                  38.16
P014                     12              319.7                  26.64
P010                     15              254.0                  16.93
P019                     15              308.3                  20.55
P023                     18              263.8                  14.66
P021                     20              360.8                  18.04
P004                     20              308.5                  15.42
P024                     22              331.2                  15.05
P009                     25              327.2                  13.09
P011                     25              329.3                  13.17
P030                     25              288.2                  11.53
P018                     30              401.5          

In [5]:
for name, df in [('po_h', po_h), ('po_l', po_l), ('so_h', so_h), ('so_l', so_l), ('ledger', ledger)]:
    print(name, df.shape)
    print(list(df.columns))
    print(df.head(2).to_string())
    print()

po_h (24000, 10)
['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']
       po_id supplier_id branch_id  order_date expected_delivery_date received_date po_status   total_cost  total_gst_amount   grand_total
0  PO-141186     SUP0001    PUN001  2019-04-10             2019-04-29    2019-04-29  Received  22955232.11      6.427465e+06  2.938270e+07
1  PO-215234     SUP0005    DEL001  2020-12-30             2021-01-27    2021-01-27  Received  19406544.39      5.428358e+06  2.483490e+07

po_l (155495, 9)
['po_id', 'line_number', 'product_id', 'quantity', 'unit_cost', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']
       po_id  line_number product_id  quantity  unit_cost  gst_rate  line_total    gst_amount  line_grand_total
0  PO-141186            1       P004       234   34709.54        28  8122032.36  2.274169e+06      1.039620e+07
1  PO-141186            2       P022       

In [6]:
sd = so_l.merge(so_h[['so_id', 'branch_id', 'order_status', 'delivery_date']], on='so_id')
sd = sd[sd['order_status'] == 'Delivered']

pr = po_l.merge(po_h[['po_id', 'branch_id', 'po_status', 'received_date']], on='po_id')
pr = pr[pr['po_status'] == 'Received']

print('sd (Delivered sales lines) rows:', len(sd), '| units delivered:', sd['quantity'].sum())
print('pr (Received purchase lines) rows:', len(pr), '| units received:', pr['quantity'].sum())

sd (Delivered sales lines) rows: 117656 | units delivered: 1234939
pr (Received purchase lines) rows: 140077 | units received: 22387123


In [7]:
L = ledger.sort_values(['product_id', 'branch_id', 'movement_date'], kind='stable').copy()
L['prev'] = L.groupby(['product_id', 'branch_id'])['running_balance'].shift(1)
op = inv.set_index(['product_id', 'branch_id'])['opening_stock']
first = L['prev'].isna()
L.loc[first, 'prev'] = [op[(p, b)] for p, b in zip(L.loc[first, 'product_id'], L.loc[first, 'branch_id'])]
L['change'] = L['running_balance'] - L['prev']

print('ledger rows:', len(L))
print('IN rows with change = +quantity:', ((L['movement_type'] == 'IN') & (L['change'] == L['quantity'])).sum(), 'of', (L['movement_type'] == 'IN').sum())
print('OUT rows with change = -quantity:', ((L['movement_type'] == 'OUT') & (L['change'] == -L['quantity'])).sum(), 'of', (L['movement_type'] == 'OUT').sum())
print('ADJUSTMENT rows with change = +/- quantity:', ((L['movement_type'] == 'ADJUSTMENT') & (L['change'].abs() == L['quantity'])).sum(), 'of', (L['movement_type'] == 'ADJUSTMENT').sum())

adj = L[L['movement_type'] == 'ADJUSTMENT'].groupby(['product_id', 'branch_id'])['change'].sum()

pair = inv[['product_id', 'branch_id', 'opening_stock']].set_index(['product_id', 'branch_id'])
pair['received'] = pr.groupby(['product_id', 'branch_id'])['quantity'].sum()
pair['delivered'] = sd.groupby(['product_id', 'branch_id'])['quantity'].sum()
pair['adjust'] = adj
pair = pair.fillna(0)
pair['stock_end'] = pair['opening_stock'] + pair['received'] - pair['delivered'] + pair['adjust']

print()
print('pair table rows:', len(pair))
print('opening:', pair['opening_stock'].sum(), '| received:', pair['received'].sum(),
      '| delivered:', pair['delivered'].sum(), '| adjustments:', pair['adjust'].sum())
print('stock_end total:', pair['stock_end'].sum(), '(KPI 8 order-line stock: 21185215)')
print('pairs with stock_end below zero:', (pair['stock_end'] < 0).sum())
print('stock_end smallest / median / largest:', pair['stock_end'].min(), pair['stock_end'].median(), pair['stock_end'].max())

ledger rows: 237230
IN rows with change = +quantity: 127611 of 127611
OUT rows with change = -quantity: 107165 of 107165
ADJUSTMENT rows with change = +/- quantity: 2454 of 2454

pair table rows: 180
opening: 32969 | received: 22387123 | delivered: 1234939 | adjustments: 62.0
stock_end total: 21185215.0 (KPI 8 order-line stock: 21185215)
pairs with stock_end below zero: 0
stock_end smallest / median / largest: 98642.0 117900.5 132967.0


In [8]:
sd['delivery_date'] = pd.to_datetime(sd['delivery_date'])
print('latest delivery date in sd:', sd['delivery_date'].max().date())

last12 = sd[(sd['delivery_date'] >= '2024-01-15') & (sd['delivery_date'] <= '2025-01-13')]
print('sd rows in the 12-month window:', len(last12), '| units:', last12['quantity'].sum())

pair['sold_12m'] = last12.groupby(['product_id', 'branch_id'])['quantity'].sum()
pair['sold_12m'] = pair['sold_12m'].fillna(0)
pair['cover_years'] = pair['stock_end'] / pair['sold_12m']

print('pairs:', len(pair), '| pairs with no sales in the window:', (pair['sold_12m'] == 0).sum())
print('units sold per pair: mean', round(pair['sold_12m'].mean(), 2),
      '| smallest', pair['sold_12m'].min(), '| largest', pair['sold_12m'].max())
print('years of cover: mean', round(pair['cover_years'].mean(), 2),
      '| smallest', round(pair['cover_years'].min(), 1), '| largest', round(pair['cover_years'].max(), 1))

latest delivery date in sd: 2025-01-13
sd rows in the 12-month window: 18889 | units: 197300
pairs: 180 | pairs with no sales in the window: 0
units sold per pair: mean 1096.11 | smallest 703 | largest 1548
years of cover: mean 110.46 | smallest 72.0 | largest 171.9


In [9]:
pair['inv_max'] = inv.set_index(['product_id', 'branch_id'])['max_stock']
pair['prod_max'] = pair.index.get_level_values('product_id').map(products.set_index('product_id')['max_stock_level'])
pair['monthly_sales'] = pair['sold_12m'] / 12

rules = {
    '1 month of sales': pair['monthly_sales'] * 1,
    '3 months of sales': pair['monthly_sales'] * 3,
    '6 months of sales': pair['monthly_sales'] * 6,
    '12 months of sales': pair['monthly_sales'] * 12,
    'company limit: inventory master max_stock': pair['inv_max'],
    'company limit: products table max_stock_level': pair['prod_max'],
}

total_stock = pair['stock_end'].sum()
print('pairs:', len(pair), '| total stock:', total_stock)
print()
for name, needed in rules.items():
    excess = (pair['stock_end'] - needed).clip(lower=0)
    print(name, '| needed units:', round(needed.sum()), '| excess units:', round(excess.sum()),
          '| excess share of stock:', round(100 * excess.sum() / total_stock, 2), '%',
          '| pairs with any excess:', (excess > 0).sum())

pairs: 180 | total stock: 21185215.0

1 month of sales | needed units: 16442 | excess units: 21168773 | excess share of stock: 99.92 % | pairs with any excess: 180
3 months of sales | needed units: 49325 | excess units: 21135890 | excess share of stock: 99.77 % | pairs with any excess: 180
6 months of sales | needed units: 98650 | excess units: 21086565 | excess share of stock: 99.53 % | pairs with any excess: 180
12 months of sales | needed units: 197300 | excess units: 20987915 | excess share of stock: 99.07 % | pairs with any excess: 180
company limit: inventory master max_stock | needed units: 57916 | excess units: 21127299 | excess share of stock: 99.73 % | pairs with any excess: 180
company limit: products table max_stock_level | needed units: 18870 | excess units: 21166345 | excess share of stock: 99.91 % | pairs with any excess: 180


In [10]:
cw = pr.groupby('product_id')['line_total'].sum() / pr.groupby('product_id')['quantity'].sum()
ct = products.set_index('product_id')['unit_cost']
pid = pair.index.get_level_values('product_id')
pair['cost_received'] = pid.map(cw)
pair['cost_table'] = pid.map(ct)

print('products with a received cost:', cw.notna().sum(), '| pairs:', len(pair))
print('received cost per unit: smallest', round(cw.min(), 2), '| largest', round(cw.max(), 2))

pair['needed_3m'] = pair['monthly_sales'] * 3
pair['excess_3m'] = (pair['stock_end'] - pair['needed_3m']).clip(lower=0)

stock_value_rc = (pair['stock_end'] * pair['cost_received']).sum()
excess_value_rc = (pair['excess_3m'] * pair['cost_received']).sum()
stock_value_ct = (pair['stock_end'] * pair['cost_table']).sum()
excess_value_ct = (pair['excess_3m'] * pair['cost_table']).sum()

print()
print('Received cost  | stock value:', round(stock_value_rc), '| excess value (3 months rule):', round(excess_value_rc),
      '|', round(100 * excess_value_rc / stock_value_rc, 2), '% of stock value')
print('Products table | stock value:', round(stock_value_ct), '| excess value (3 months rule):', round(excess_value_ct),
      '|', round(100 * excess_value_ct / stock_value_ct, 2), '% of stock value')

products with a received cost: 30 | pairs: 180
received cost per unit: smallest 232.45 | largest 89239.25

Received cost  | stock value: 295255911357 | excess value (3 months rule): 294576746335 | 99.77 % of stock value
Products table | stock value: 308895127500 | excess value (3 months rule): 308184841342 | 99.77 % of stock value


In [11]:
pair['excess_value'] = pair['excess_3m'] * pair['cost_received']
pair['stock_value'] = pair['stock_end'] * pair['cost_received']

by_branch = pair.groupby('branch_id')[['stock_value', 'excess_value']].sum()
by_branch['share_of_excess_pct'] = (100 * by_branch['excess_value'] / by_branch['excess_value'].sum()).round(2)
show = by_branch.copy()
show['stock_value'] = show['stock_value'].round(0).astype('int64')
show['excess_value'] = show['excess_value'].round(0).astype('int64')
print('By branch (received cost, 3 months rule):')
print(show.to_string())
print('sum of the branches (excess value):', round(by_branch['excess_value'].sum()))

names = products.set_index('product_id')['product_name']
by_prod = pair.groupby('product_id')['excess_value'].sum().sort_values(ascending=False)
top = by_prod.head(5)
low = by_prod.tail(5)
print()
print('Top 5 products by excess value:')
print(pd.DataFrame({'product': names[top.index].values, 'excess_value': top.round(0).astype('int64').values}, index=top.index).to_string())
print('top 5 share of total excess:', round(100 * top.sum() / by_prod.sum(), 2), '%')
print()
print('Smallest 5 products by excess value:')
print(pd.DataFrame({'product': names[low.index].values, 'excess_value': low.round(0).astype('int64').values}, index=low.index).to_string())

spend = pr['line_total'].sum()
print()
print('total spent on received purchases:', round(spend))
print('excess value as % of that spend:', round(100 * pair['excess_value'].sum() / spend, 2), '%')

By branch (received cost, 3 months rule):
           stock_value  excess_value  share_of_excess_pct
branch_id                                                
AHM001     48651160842   48543863142                16.48
CHN001     50072886813   49935378833                16.95
DEL001     49928692929   49823387369                16.91
HYD001     49509223586   49413003293                16.77
KOL001     47098343177   46967415811                15.94
PUN001     49995604011   49893697886                16.94
sum of the branches (excess value): 294576746335

Top 5 products by excess value:
                                 product  excess_value
product_id                                            
P015                 Engine Block EB-600   61841619704
P014        Transmission Assembly TA-800   45331993198
P021                Boom Cylinder BC-400   24247416699
P004                   Track Chain TC-45   20808955672
P019                 Swing Motor SWM-350   20330241141
top 5 share of total excess

In [12]:
idx = pair.index
cols = list(range(2019, 2026))

rec = (pr.assign(year=pd.to_datetime(pr['received_date']).dt.year)
         .groupby(['product_id', 'branch_id', 'year'])['quantity'].sum()
         .unstack(fill_value=0).reindex(index=idx, columns=cols, fill_value=0))
dele = (sd.assign(year=sd['delivery_date'].dt.year)
          .groupby(['product_id', 'branch_id', 'year'])['quantity'].sum()
          .unstack(fill_value=0).reindex(index=idx, columns=cols, fill_value=0))
a = L[L['movement_type'] == 'ADJUSTMENT'].copy()
a['year'] = pd.to_datetime(a['movement_date']).dt.year
adjy = (a.groupby(['product_id', 'branch_id', 'year'])['change'].sum()
          .unstack(fill_value=0).reindex(index=idx, columns=cols, fill_value=0))

stock_y = (rec - dele + adjy).cumsum(axis=1).add(pair['opening_stock'], axis=0)

rows = []
for y in range(2019, 2025):
    needed = dele[y] / 12 * 3
    exc = (stock_y[y] - needed).clip(lower=0)
    rows.append({'year_end': y,
                 'stock_units': int(stock_y[y].sum()),
                 'excess_units': int(exc.sum()),
                 'excess_value': int((exc * pair['cost_received']).sum())})
trend = pd.DataFrame(rows).set_index('year_end')
print(trend.to_string())
print()
print('stock units at the end of Jan 2025 (from this rebuild):', int(stock_y[2025].sum()), '| expected 21185215')


          stock_units  excess_units  excess_value
year_end                                         
2019          3384660       3333513   46509105934
2020          6960247       6907449   96005272745
2021         10529943      10478209  145745513777
2022         14009405      13958726  194159992126
2023         17557387      17505992  244028606230
2024         20993633      20943492  291825795283

stock units at the end of Jan 2025 (from this rebuild): 21185215 | expected 21185215


In [13]:
months = pd.period_range('2019-01', '2025-01', freq='M')

def monthly(df, date_col, val_col):
    t = df.copy()
    t['m'] = pd.to_datetime(t[date_col]).dt.to_period('M')
    return (t.groupby(['product_id', 'branch_id', 'm'])[val_col].sum()
             .unstack(fill_value=0).reindex(index=pair.index, columns=months, fill_value=0))

rec_m = monthly(pr, 'received_date', 'quantity')
del_m = monthly(sd, 'delivery_date', 'quantity')
adj_m = monthly(L[L['movement_type'] == 'ADJUSTMENT'], 'movement_date', 'change')
stock_m = (rec_m - del_m + adj_m).cumsum(axis=1).add(pair['opening_stock'], axis=0)

reorder = inv.set_index(['product_id', 'branch_id'])['reorder_level']
safety = inv.set_index(['product_id', 'branch_id'])['safety_stock']
below_reorder = stock_m.lt(reorder, axis=0)
below_safety = stock_m.lt(safety, axis=0)

print('pairs:', len(stock_m), '| months:', stock_m.shape[1], '| pair-months:', stock_m.size)
print('total stock at the end of Jan 2025:', int(stock_m.iloc[:, -1].sum()), '| expected 21185215')
print('smallest month-end stock of any pair:', int(stock_m.min().min()), '| expected 67')
print('pair-months below reorder_level:', int(below_reorder.sum().sum()), '| pairs affected:', int(below_reorder.any(axis=1).sum()))
print('pair-months below safety_stock:', int(below_safety.sum().sum()), '| pairs affected:', int(below_safety.any(axis=1).sum()))

pairs: 180 | months: 73 | pair-months: 13140
total stock at the end of Jan 2025: 21185215 | expected 21185215
smallest month-end stock of any pair: 67 | expected 67
pair-months below reorder_level: 0 | pairs affected: 0
pair-months below safety_stock: 0 | pairs affected: 0


In [14]:
pt_reorder = pd.Series(pair.index.get_level_values('product_id').map(products.set_index('product_id')['reorder_level']), index=pair.index)
pt_safety = pd.Series(pair.index.get_level_values('product_id').map(products.set_index('product_id')['safety_stock']), index=pair.index)

below_pt_reorder = stock_m.lt(pt_reorder, axis=0)
below_pt_safety = stock_m.lt(pt_safety, axis=0)
print('products table: pair-months below reorder_level:', int(below_pt_reorder.sum().sum()), '| pairs affected:', int(below_pt_reorder.any(axis=1).sum()))
print('products table: pair-months below safety_stock:', int(below_pt_safety.sum().sum()), '| pairs affected:', int(below_pt_safety.any(axis=1).sum()))

ratio = stock_m.div(reorder, axis=0)
print()
print('closest any pair came to its inventory master reorder level (stock / reorder_level):', round(ratio.min().min(), 2))

print()
print('5 lowest month-end stocks (product, branch, month):')
print(stock_m.stack().sort_values().head(5).to_string())

products table: pair-months below reorder_level: 0 | pairs affected: 0
products table: pair-months below safety_stock: 0 | pairs affected: 0

closest any pair came to its inventory master reorder level (stock / reorder_level): 1.91

5 lowest month-end stocks (product, branch, month):
product_id  branch_id         
P016        HYD001     2019-01     67.0
P004        HYD001     2019-01    132.0
P011        AHM001     2019-01    149.0
P025        HYD001     2019-01    190.0
P023        HYD001     2019-01    230.0


In [15]:
cols = ['product_id', 'branch_id']
ev = pd.concat([
    pr.assign(d=pd.to_datetime(pr['received_date']), chg=pr['quantity'])[cols + ['d', 'chg']],
    sd.assign(d=pd.to_datetime(sd['delivery_date']), chg=-sd['quantity'])[cols + ['d', 'chg']],
    L[L['movement_type'] == 'ADJUSTMENT'].assign(d=pd.to_datetime(L['movement_date']), chg=L['change'])[cols + ['d', 'chg']]
])
daily = ev.groupby(cols + ['d'])['chg'].sum()
bal = daily.groupby(level=[0, 1]).cumsum() + pair['opening_stock'].reindex(daily.index.droplevel('d')).values

last = bal.groupby(level=[0, 1]).last().reindex(pair.index)
low = bal.groupby(level=[0, 1]).min().reindex(pair.index)

print('dtype of d:', ev['d'].dtype, '| event rows:', len(ev), '| pair-days:', len(bal))
print('last daily balance = stock_end for', int((last == pair['stock_end']).sum()), 'of 180 pairs')
print('lowest end-of-day stock of any pair:', low.min())
print('pairs that dipped below inventory-master reorder_level:', int((low < reorder.reindex(pair.index)).sum()), '| safety_stock:', int((low < safety.reindex(pair.index)).sum()))
print('pairs that dipped below products-table reorder_level:', int((low < pt_reorder.reindex(pair.index)).sum()), '| safety_stock:', int((low < pt_safety.reindex(pair.index)).sum()))

dtype of d: datetime64[ns] | event rows: 260187 | pair-days: 175335
last daily balance = stock_end for 180 of 180 pairs
lowest end-of-day stock of any pair: 0.0
pairs that dipped below inventory-master reorder_level: 5 | safety_stock: 1
pairs that dipped below products-table reorder_level: 11 | safety_stock: 6


In [16]:
lv = pd.DataFrame({'inv_reorder': reorder, 'inv_safety': safety, 'pt_reorder': pt_reorder, 'pt_safety': pt_safety}).reindex(pair.index)
lv['highest_level'] = lv.max(axis=1)
b = bal.rename('stock').reset_index().merge(lv.reset_index(), on=cols)
hit = b[b['stock'] < b['highest_level']]

print('pair-days below at least one level:', len(hit), '| pairs:', hit.groupby(cols).ngroups)
print(hit.groupby(cols).agg(days=('d', 'count'), first_day=('d', 'min'), last_day=('d', 'max'), lowest=('stock', 'min'), highest_level=('highest_level', 'first')))

pair-days below at least one level: 42 | pairs: 14
                      days  first_day   last_day  lowest  highest_level
product_id branch_id                                                   
P004       KOL001        1 2019-01-12 2019-01-12    37.0             44
P005       CHN001        1 2019-01-16 2019-01-16    62.0             68
P008       PUN001        3 2019-01-03 2019-01-12    34.0            120
P013       PUN001        6 2019-01-04 2019-01-15    19.0             80
P020       DEL001        2 2019-01-11 2019-01-17    67.0             90
           PUN001        1 2019-01-07 2019-01-07    75.0             90
P023       HYD001        2 2019-01-09 2019-01-12    22.0             31
P026       CHN001        5 2019-01-07 2019-01-20    73.0            200
           DEL001        2 2019-01-06 2019-01-12    94.0            200
           HYD001        7 2019-01-07 2019-01-25   148.0            200
P027       AHM001        2 2019-01-05 2019-01-12   135.0            150
           CH

In [17]:
def scan_text(name, df, columns):
    print(name, '| rows:', len(df))
    for c in columns:
        s = df[c]
        print('   ', c, '| empty:', int(s.isna().sum() + (s.astype(str).str.strip() == '').sum()))

def scan_num(name, df, columns):
    for c in columns:
        s = df[c]
        print('   ', c, '| empty:', int(s.isna().sum()), '| zeros:', int((s == 0).sum()), '| negatives:', int((s < 0).sum()))

scan_text('sd (delivered sales lines)', sd, ['product_id', 'branch_id', 'delivery_date']); scan_num('', sd, ['quantity'])
scan_text('pr (received purchase lines)', pr, ['product_id', 'branch_id', 'received_date']); scan_num('', pr, ['quantity', 'line_total'])
scan_text('inv (inventory master)', inv, ['product_id', 'branch_id']); scan_num('', inv, ['opening_stock', 'reorder_level', 'safety_stock', 'max_stock'])
scan_text('products', products, ['product_id']); scan_num('', products, ['unit_cost', 'reorder_level', 'safety_stock', 'max_stock_level'])
scan_text('L (ledger)', L, ['product_id', 'branch_id', 'movement_type', 'movement_date']); scan_num('', L, ['quantity', 'running_balance'])
pr_ = pair.reset_index()
scan_text('pair', pr_, ['product_id', 'branch_id'])
scan_num('', pr_, ['opening_stock', 'received', 'delivered', 'stock_end', 'sold_12m', 'cover_years', 'inv_max', 'prod_max', 'monthly_sales', 'cost_received', 'cost_table', 'needed_3m', 'excess_3m', 'excess_value', 'stock_value'])
print('    adjust | empty:', int(pr_['adjust'].isna().sum()))

sd (delivered sales lines) | rows: 117656
    product_id | empty: 0
    branch_id | empty: 0
    delivery_date | empty: 0
    quantity | empty: 0 | zeros: 0 | negatives: 0
pr (received purchase lines) | rows: 140077
    product_id | empty: 0
    branch_id | empty: 0
    received_date | empty: 0
    quantity | empty: 0 | zeros: 0 | negatives: 0
    line_total | empty: 0 | zeros: 0 | negatives: 0
inv (inventory master) | rows: 180
    product_id | empty: 0
    branch_id | empty: 0
    opening_stock | empty: 0 | zeros: 0 | negatives: 0
    reorder_level | empty: 0 | zeros: 0 | negatives: 0
    safety_stock | empty: 0 | zeros: 0 | negatives: 0
    max_stock | empty: 0 | zeros: 0 | negatives: 0
products | rows: 30
    product_id | empty: 0
    unit_cost | empty: 0 | zeros: 0 | negatives: 0
    reorder_level | empty: 0 | zeros: 0 | negatives: 0
    safety_stock | empty: 0 | zeros: 0 | negatives: 0
    max_stock_level | empty: 0 | zeros: 0 | negatives: 0
L (ledger) | rows: 237230
    product_

In [18]:
txt = pd.concat([
    pr[cols + ['received_date']].rename(columns={'received_date': 'd'}),
    L[L['movement_type'] == 'ADJUSTMENT'][cols + ['movement_date']].rename(columns={'movement_date': 'd'})
])
n_text = len(txt.drop_duplicates())
n_sd = len(sd[cols + ['delivery_date']].drop_duplicates())
print('distinct text pair-days (receipts + adjustments):', n_text)
print('distinct sales pair-days:', n_sd)
print('text + sd:', n_text + n_sd)
print('correct pair-days:', len(daily), '| same day in both:', n_text + n_sd - len(daily))

distinct text pair-days (receipts + adjustments): 108505
distinct sales pair-days: 91644
text + sd: 200149
correct pair-days: 175335 | same day in both: 24814


In [19]:
rd = pr.assign(d=pd.to_datetime(pr['received_date']))
g = hit.groupby(cols).agg(first_dip=('d', 'min'), last_dip=('d', 'max'), lowest=('stock', 'min')).reset_index()
g = g.merge(inv[cols + ['opening_stock']], on=cols)
g['first_receipt'] = [rd[(rd['product_id'] == p) & (rd['branch_id'] == b)]['d'].min() for p, b in zip(g['product_id'], g['branch_id'])]
g['receipts_up_to_last_dip'] = [int(((rd['product_id'] == p) & (rd['branch_id'] == b) & (rd['d'] <= ld)).sum()) for p, b, ld in zip(g['product_id'], g['branch_id'], g['last_dip'])]
print(g)

   product_id branch_id  first_dip   last_dip  lowest  opening_stock  \
0        P004    KOL001 2019-01-12 2019-01-12    37.0            115   
1        P005    CHN001 2019-01-16 2019-01-16    62.0            172   
2        P008    PUN001 2019-01-03 2019-01-12    34.0             86   
3        P013    PUN001 2019-01-04 2019-01-15    19.0             80   
4        P020    DEL001 2019-01-11 2019-01-17    67.0             91   
5        P020    PUN001 2019-01-07 2019-01-07    75.0             80   
6        P023    HYD001 2019-01-09 2019-01-12    22.0             82   
7        P026    CHN001 2019-01-07 2019-01-20    73.0            113   
8        P026    DEL001 2019-01-06 2019-01-12    94.0             96   
9        P026    HYD001 2019-01-07 2019-01-25   148.0            203   
10       P027    AHM001 2019-01-05 2019-01-12   135.0            151   
11       P027    CHN001 2019-01-03 2019-01-17     0.0             84   
12       P027    DEL001 2019-01-05 2019-01-20    62.0           

In [20]:
p, b = 'P027', 'HYD001'
print(ev[(ev['product_id'] == p) & (ev['branch_id'] == b)].sort_values('d').head(8))
print(bal.loc[(p, b)].head(8))

       product_id branch_id          d    chg
209746       P027    HYD001 2019-01-02   20.0
48014        P027    HYD001 2019-01-21   -2.0
100108       P027    HYD001 2019-01-25  233.0
19876        P027    HYD001 2019-01-27   92.0
49378        P027    HYD001 2019-01-28  159.0
105905       P027    HYD001 2019-01-29  261.0
68854        P027    HYD001 2019-02-01  205.0
108621       P027    HYD001 2019-02-06   -8.0
d
2019-01-02     143.0
2019-01-21     141.0
2019-01-25     374.0
2019-01-27     466.0
2019-01-28     625.0
2019-01-29     886.0
2019-02-01    1091.0
2019-02-06    1148.0
Name: chg, dtype: float64


In [21]:
x = L[(L['product_id'] == 'P027') & (L['branch_id'] == 'HYD001')].sort_values('movement_date')
print(x[['movement_date', 'movement_type', 'quantity', 'running_balance']].head(6).to_string(index=False))

movement_date movement_type  quantity  running_balance
   2019-01-02    ADJUSTMENT        20              143
   2019-01-21           OUT         2              141
   2019-01-27            IN        92              233
   2019-01-28            IN       159              392
   2019-01-29            IN       261              653
   2019-02-01            IN       205              858


In [22]:
lgd = L.assign(d=pd.to_datetime(L['movement_date'])).groupby(cols + ['d'])['running_balance'].last()
lowL = lgd.groupby(level=[0, 1]).min().reindex(pair.index)
print('ledger pair-days:', len(lgd), '| lowest end-of-day ledger balance of any pair:', lowL.min())
print('pairs below inventory-master reorder_level:', int((lowL < reorder.reindex(pair.index)).sum()), '| safety_stock:', int((lowL < safety.reindex(pair.index)).sum()))
print('pairs below products-table reorder_level:', int((lowL < pt_reorder.reindex(pair.index)).sum()), '| safety_stock:', int((lowL < pt_safety.reindex(pair.index)).sum()))
bL = lgd.rename('stock').reset_index().merge(lv.reset_index(), on=cols)
hitL = bL[bL['stock'] < bL['highest_level']]
print('pair-days below at least one level:', len(hitL), '| pairs:', hitL.groupby(cols).ngroups, '| first day:', hitL['d'].min().date(), '| last day:', hitL['d'].max().date())

ledger pair-days: 165246 | lowest end-of-day ledger balance of any pair: 14
pairs below inventory-master reorder_level: 3 | safety_stock: 1
pairs below products-table reorder_level: 11 | safety_stock: 6
pair-days below at least one level: 37 | pairs: 13 | first day: 2019-01-02 | last day: 2019-01-28


In [23]:
three = g[g['receipts_up_to_last_dip'] > 0]
for p, b, fr, ld in zip(three['product_id'], three['branch_id'], three['first_receipt'], three['last_dip']):
    print(p, b, '| highest level:', lv.loc[(p, b), 'highest_level'])
    print(rd[(rd['product_id'] == p) & (rd['branch_id'] == b) & (rd['d'] <= ld)][['d', 'quantity']].to_string(index=False))
    print(bal.loc[(p, b)].loc[fr:ld].to_string())


P008 PUN001 | highest level: 120
         d  quantity
2019-01-12        84
d
2019-01-12    118.0
P013 PUN001 | highest level: 80
         d  quantity
2019-01-12        39
d
2019-01-12    58.0
2019-01-15    62.0
P026 DEL001 | highest level: 200
         d  quantity
2019-01-12        85
d
2019-01-12    172.0


In [24]:
win = sd[(pd.to_datetime(sd['delivery_date']) >= '2024-01-15') & (pd.to_datetime(sd['delivery_date']) <= '2025-01-13')]
lines_win = win.groupby(cols).size().reindex(pair.index, fill_value=0)
print('window delivered lines:', len(win), '| units:', int(win['quantity'].sum()), '| sum of sold_12m:', int(pair['sold_12m'].sum()))
print('pairs with at least one delivered line in the window:', int((lines_win > 0).sum()), 'of', len(pair), '| fewest lines in one pair:', int(lines_win.min()))
print('smallest needed_3m:', round(pair['needed_3m'].min(), 2), '| pairs where needed_3m > stock_end:', int((pair['needed_3m'] > pair['stock_end']).sum()))
print('pairs with negative stock_end:', int((pair['stock_end'] < 0).sum()), '| lowest end-of-day stock of any pair:', low.min())
rp = pr.groupby('product_id')['quantity'].sum()
print('products with received lines:', len(rp), 'of', len(products), '| smallest cost_received:', round(pair['cost_received'].min(), 2), '| pairs with cost_received <= 0:', int((pair['cost_received'] <= 0).sum()))

window delivered lines: 18889 | units: 197300 | sum of sold_12m: 197300
pairs with at least one delivered line in the window: 180 of 180 | fewest lines in one pair: 67
smallest needed_3m: 175.75 | pairs where needed_3m > stock_end: 0
pairs with negative stock_end: 0 | lowest end-of-day stock of any pair: 0.0
products with received lines: 30 of 30 | smallest cost_received: 232.45 | pairs with cost_received <= 0: 0


In [25]:
for name, df, oid in [('sd', sd, 'so_id'), ('pr', pr, 'po_id')]:
    other_cols = [c for c in df.columns if c != 'line_number']
    print(name, '| rows:', len(df),
          '| duplicate (order id, line_number):', int(df.duplicated([oid, 'line_number']).sum()),
          '| lines sharing an order and product:', int(df.duplicated([oid, 'product_id'], keep=False).sum()),
          '| exact copies (all columns the same except line_number):', int(df.duplicated(other_cols).sum()))
print('pair rows:', len(pair), '| pair unique:', pair.index.is_unique)

sd | rows: 117656 | duplicate (order id, line_number): 0 | lines sharing an order and product: 25422 | exact copies (all columns the same except line_number): 688
pr | rows: 140077 | duplicate (order id, line_number): 0 | lines sharing an order and product: 30068 | exact copies (all columns the same except line_number): 0
pair rows: 180 | pair unique: True


In [26]:
g = sd.groupby(['so_id', 'product_id']).size().rename('n').reset_index()
g = g[g['n'] >= 2]
g['pairs'] = g['n'] * (g['n'] - 1) / 2
s = sd.groupby('product_id')['quantity'].apply(lambda q: float((q.value_counts(normalize=True) ** 2).sum()))
expected = float((g['pairs'] * g['product_id'].map(s)).sum())
other_cols = [c for c in sd.columns if c != 'line_number']
observed = float(sd.groupby(other_cols).size().pipe(lambda x: x * (x - 1) / 2).sum())
print('order-product groups with 2+ lines:', len(g), '| line pairs inside them:', int(g['pairs'].sum()))
print('pairs that are exact copies:', int(observed), '| expected by chance:', round(expected, 1))

order-product groups with 2+ lines: 12184 | line pairs inside them: 14369
pairs that are exact copies: 690 | expected by chance: 721.7


In [27]:
tot = lambda c: int(round(pair[c].sum()))
print('opening + received - delivered + adjust =', tot('opening_stock') + tot('received') - tot('delivered') + tot('adjust'), '| stock_end total:', tot('stock_end'), '| expected: 21185215')
print('received:', tot('received'), '| expected: 22387123 | delivered:', tot('delivered'), '| expected: 1234939 | opening:', tot('opening_stock'), '| expected: 32969 | adjust:', tot('adjust'), '| expected: 62')
print('largest gap between needed_3m + excess_3m and stock_end in any pair:', round((pair['needed_3m'] + pair['excess_3m'] - pair['stock_end']).abs().max(), 6), '| needed_3m total:', round(pair['needed_3m'].sum(), 2), '| excess_3m total:', tot('excess_3m'), '| expected: 21135890')
excess_check = pair['excess_3m'] * pair['cost_received']
print('excess value, column total:', tot('excess_value'), '| units x cost recomputed:', int(round(excess_check.sum())), '| expected: 294576746335')
print('sum of branch totals:', int(round(pair['excess_value'].groupby(level='branch_id').sum().sum())), '| sum of product totals:', int(round(pair['excess_value'].groupby(level='product_id').sum().sum())))
print('stock value:', tot('stock_value'), '| expected: 295255911357 | spend on received purchases (pr line_total):', int(round(pr['line_total'].sum())), '| expected: 312027429108')

opening + received - delivered + adjust = 21185215 | stock_end total: 21185215 | expected: 21185215
received: 22387123 | expected: 22387123 | delivered: 1234939 | expected: 1234939 | opening: 32969 | expected: 32969 | adjust: 62 | expected: 62
largest gap between needed_3m + excess_3m and stock_end in any pair: 0.0 | needed_3m total: 49325.0 | excess_3m total: 21135890 | expected: 21135890
excess value, column total: 294576746335 | units x cost recomputed: 294576746335 | expected: 294576746335
sum of branch totals: 294576746335 | sum of product totals: 294576746335
stock value: 295255911357 | expected: 295255911357 | spend on received purchases (pr line_total): 312027429108 | expected: 312027429108


In [28]:
rec2 = pr.pivot_table(index=cols, values='quantity', aggfunc='sum').rename(columns={'quantity': 'rec'})
del2 = sd.pivot_table(index=cols, values='quantity', aggfunc='sum').rename(columns={'quantity': 'dele'})
adj2 = L[L['movement_type'] == 'ADJUSTMENT'].pivot_table(index=cols, values='change', aggfunc='sum').rename(columns={'change': 'adj'})
in_window = pd.to_datetime(sd['delivery_date']).between('2024-01-15', '2025-01-13')
sold2 = sd[in_window].pivot_table(index=cols, values='quantity', aggfunc='sum').rename(columns={'quantity': 'sold'})
x = inv.set_index(cols)[['opening_stock']].join(rec2).join(del2).join(adj2).join(sold2).fillna(0)
x['stock'] = x['opening_stock'] + x['rec'] - x['dele'] + x['adj']
x['excess'] = (x['stock'] - 3 * x['sold'] / 12).clip(lower=0)
cost = pr.groupby('product_id')['line_total'].sum() / pr.groupby('product_id')['quantity'].sum()
x['cost'] = x.index.get_level_values('product_id').map(cost)
print('pairs:', len(x), '| stock total:', int(round(x['stock'].sum())), '| expected: 21185215')
print('excess units:', int(round(x['excess'].sum())), '| expected: 21135890')
print('excess value:', int(round((x['excess'] * x['cost']).sum())), '| expected: 294576746335')
print('largest difference to the pair table, stock:', x['stock'].sub(pair['stock_end']).abs().max(), '| excess units:', x['excess'].sub(pair['excess_3m']).abs().max())

pairs: 180 | stock total: 21185215 | expected: 21185215
excess units: 21135890 | expected: 21135890
excess value: 294576746335 | expected: 294576746335
largest difference to the pair table, stock: 0.0 | excess units: 0.0


In [29]:
led_stock = L.groupby(cols)['running_balance'].last().reindex(pair.index)
exL = (led_stock - pair['needed_3m']).clip(lower=0)
valL = (exL * pair['cost_received']).sum()
print('ledger stock total:', int(led_stock.sum()), '| expected: 19305994')
print('pairs where needed_3m > ledger stock:', int((pair['needed_3m'] > led_stock).sum()))
print('excess units on ledger stock:', int(exL.sum()), '| share of ledger stock:', round(100 * exL.sum() / led_stock.sum(), 2), '%')
print('excess value on ledger stock:', int(round(valL)), '| order-line headline: 294576746335 | ledger value is', round(100 * valL / 294576746335, 2), '% of the headline')

ledger stock total: 19305994 | expected: 19305994
pairs where needed_3m > ledger stock: 0
excess units on ledger stock: 19256669 | share of ledger stock: 99.74 %
excess value on ledger stock: 268214199098 | order-line headline: 294576746335 | ledger value is 91.05 % of the headline


In [30]:
for c in ['excess_3m', 'excess_value']:
    s = pair[c]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    print(c, '| q1:', round(q1), '| q3:', round(q3), '| normal range:', round(lo), 'to', round(hi), '| pairs below:', int((s < lo).sum()), '| pairs above:', int((s > hi).sum()))
show = ['excess_3m', 'excess_value', 'cost_received', 'sold_12m']
print('5 largest pairs by excess value:')
print(pair.sort_values('excess_value', ascending=False)[show].head(5))
print('5 smallest pairs by excess value:')
print(pair.sort_values('excess_value')[show].head(5))

excess_3m | q1: 113295 | q3: 121150 | normal range: 101513 to 132932 | pairs below: 1 | pairs above: 0
excess_value | q1: 126004441 | q3: 2140896367 | normal range: -2896333448 to 5163234256 | pairs below: 0 | pairs above: 12
5 largest pairs by excess value:
                      excess_3m  excess_value  cost_received  sold_12m
product_id branch_id                                                  
P015       CHN001     125269.00  1.117891e+10   89239.252704      1192
           HYD001     120540.50  1.075694e+10   89239.252704       998
           AHM001     115641.50  1.031976e+10   89239.252704      1074
           PUN001     115333.25  1.029225e+10   89239.252704       703
           DEL001     109952.25  9.812057e+09   89239.252704       955
5 smallest pairs by excess value:
                      excess_3m  excess_value  cost_received  sold_12m
product_id branch_id                                                  
P026       DEL001     112178.75  2.607630e+07     232.453094      10

In [31]:
s = pair['excess_3m']
lo = s.quantile(0.25) - 1.5 * (s.quantile(0.75) - s.quantile(0.25))
print('Pairs below the normal range in excess_3m:', int((s < lo).sum()))
print(pair[s < lo][['stock_end', 'sold_12m', 'needed_3m', 'excess_3m']])

Pairs below the normal range in excess_3m: 1
                      stock_end  sold_12m  needed_3m  excess_3m
product_id branch_id                                           
P019       KOL001       98642.0      1009     252.25   98389.75


In [32]:
v = pair['excess_value']
hiv = v.quantile(0.75) + 1.5 * (v.quantile(0.75) - v.quantile(0.25))
out = pair[v > hiv]
print('Pairs above the normal range in excess_value:', len(out))
print(out[['excess_3m', 'excess_value', 'cost_received']].sort_values('excess_value', ascending=False))
print('Pairs per product:')
print(out.groupby(level='product_id').size())

Pairs above the normal range in excess_value: 12
                      excess_3m  excess_value  cost_received
product_id branch_id                                        
P015       CHN001     125269.00  1.117891e+10   89239.252704
           HYD001     120540.50  1.075694e+10   89239.252704
           AHM001     115641.50  1.031976e+10   89239.252704
           PUN001     115333.25  1.029225e+10   89239.252704
           DEL001     109952.25  9.812057e+09   89239.252704
           KOL001     106250.25  9.481693e+09   89239.252704
P014       CHN001     123398.50  7.738671e+09   62712.841557
           PUN001     122746.00  7.697750e+09   62712.841557
           DEL001     122678.00  7.693486e+09   62712.841557
           HYD001     121328.00  7.608824e+09   62712.841557
           KOL001     116716.75  7.319639e+09   62712.841557
           AHM001     115983.00  7.273624e+09   62712.841557
Pairs per product:
product_id
P014    6
P015    6
dtype: int64


In [33]:
cost_rank = pair.groupby(level='product_id')['cost_received'].first().sort_values(ascending=False)
print('3 highest received costs per unit:')
print(cost_rank.head(3))

rest = pair[pair['excess_value'] <= hiv]
print('Largest excess_value among the unflagged pairs:', round(rest['excess_value'].max()), '| cut-off:', round(hiv))

tot12 = out['excess_value'].sum()
print('12 flagged pairs, excess value:', round(tot12), '| share of headline %:', round(tot12 / pair['excess_value'].sum() * 100, 2))

3 highest received costs per unit:
product_id
P015    89239.252704
P014    62712.841557
P021    34558.287977
Name: cost_received, dtype: float64
Largest excess_value among the unflagged pairs: 4374068428 | cut-off: 5163234256
12 flagged pairs, excess value: 107173612901 | share of headline %: 36.38


In [34]:
p15 = pair.xs('P015', level='product_id')['excess_value'].sum()
p14 = pair.xs('P014', level='product_id')['excess_value'].sum()
print('P015 exact:', p15, '| P014 exact:', p14, '| sum:', p15 + p14)
print('Largest unflagged pair:')
print(rest['excess_value'].sort_values(ascending=False).head(1))

P015 exact: 61841619703.55168 | P014 exact: 45331993197.62757 | sum: 107173612901.17926
Largest unflagged pair:
product_id  branch_id
P021        DEL001       4.374068e+09
Name: excess_value, dtype: float64


In [35]:
cs = [str(c)[:7] for c in stock_m.columns]
print('month columns:', stock_m.shape[1], '| first:', cs[0], '| last:', cs[-1], '| pairs:', stock_m.shape[0])
for y in range(2019, 2025):
    print(y, 'year-end stock total:', int(stock_m.iloc[:, cs.index(f'{y}-12')].sum()))
print('last month total:', int(stock_m.iloc[:, -1].sum()))
print('receipts (pr) first date:', pd.to_datetime(pr['received_date']).min().date(), '| last date:', pd.to_datetime(pr['received_date']).max().date())
print('deliveries (sd) first date:', pd.to_datetime(sd['delivery_date']).min().date(), '| last date:', pd.to_datetime(sd['delivery_date']).max().date())

month columns: 73 | first: 2019-01 | last: 2025-01 | pairs: 180
2019 year-end stock total: 3384660
2020 year-end stock total: 6960247
2021 year-end stock total: 10529943
2022 year-end stock total: 14009405
2023 year-end stock total: 17557387
2024 year-end stock total: 20993633
last month total: 21185215
receipts (pr) first date: 2019-01-11 | last date: 2025-01-28
deliveries (sd) first date: 2019-01-03 | last date: 2025-01-13


In [36]:
cal = sd[(pd.to_datetime(sd['delivery_date']) >= '2024-01-01') & (pd.to_datetime(sd['delivery_date']) <= '2024-12-31')].groupby(['product_id', 'branch_id'])['quantity'].sum().reindex(pair.index)
print('pairs without calendar 2024 sales:', int(cal.isna().sum()))
cal = cal.fillna(0)
need_c = 3 * cal / 12
exc_c = (pair['stock_end'] - need_c).clip(lower=0)
val_c = (exc_c * pair['cost_received']).sum()
print('calendar 2024 units sold:', int(cal.sum()), '| window units sold: 197,300')
print('excess units, need from calendar 2024:', int(exc_c.sum()), '| headline 21,135,890')
print('excess value, need from calendar 2024:', round(val_c), '| headline 294,576,746,335')
print('difference in value:', round(val_c - pair['excess_value'].sum()), '| in % of headline:', round((val_c - pair['excess_value'].sum()) / pair['excess_value'].sum() * 100, 4))

pairs without calendar 2024 sales: 0
calendar 2024 units sold: 200562 | window units sold: 197,300
excess units, need from calendar 2024: 21135074 | headline 21,135,890
excess value, need from calendar 2024: 294565059230 | headline 294,576,746,335
difference in value: -11687105 | in % of headline: -0.004


In [37]:
dec = stock_m.iloc[:, cs.index('2024-12')]
jan = stock_m.iloc[:, -1]
jan_value = ((jan - dec) * pair['cost_received']).sum()
print('Units added in Jan 2025:', int((jan - dec).sum()), '| expected 191,582')
print('Value of stock added in Jan 2025:', round(jan_value), '| expected about 2,739,263,947')
print('Headline minus 2024 year-end excess value:', round(pair['excess_value'].sum() - 291825795283))
print('Window effect plus January stock value:', round((pair['excess_value'].sum() - val_c) + jan_value))

Units added in Jan 2025: 191582 | expected 191,582
Value of stock added in Jan 2025: 2739263947 | expected about 2,739,263,947
Headline minus 2024 year-end excess value: 2750951052
Window effect plus January stock value: 2750951052


In [38]:
picks = [('P015', 'CHN001'), ('P026', 'DEL001'), pair['excess_value'].sort_values().index[90]]
d = pd.to_datetime(sd['delivery_date'])
for p, b in picks:
    sold = sd[(sd['product_id'] == p) & (sd['branch_id'] == b) & (d >= '2024-01-15') & (d <= '2025-01-13')]['quantity'].sum()
    stock = stock_m.loc[(p, b)].iloc[-1]
    need = sold / 12 * 3
    exc = stock - need
    val = exc * pair.loc[(p, b), 'cost_received']
    print(p, b, '| sold:', sold, 'vs table', pair.loc[(p, b), 'sold_12m'], '| stock:', stock, '| need:', round(need, 2), '| excess:', round(exc, 2), 'vs table', pair.loc[(p, b), 'excess_3m'], '| value:', round(val), 'vs table', round(pair.loc[(p, b), 'excess_value']))


P015 CHN001 | sold: 1192 vs table 1192 | stock: 125567.0 | need: 298.0 | excess: 125269.0 vs table 125269.0 | value: 11178911947 vs table 11178911947
P026 DEL001 | sold: 1013 vs table 1013 | stock: 112432.0 | need: 253.25 | excess: 112178.75 vs table 112178.75 | value: 26076298 vs table 26076298
P018 KOL001 | sold: 1312 vs table 1312 | stock: 106960.0 | need: 328.0 | excess: 106632.0 vs table 106632.0 | value: 904392890 vs table 904392890
